In [1]:
import pandas as pd
import camelot

# 1. pages='all' 會抓取 PDF 內所有偵測到的表格 
path = R"/Users/tonylung/Desktop/上銀滾珠螺桿 FDC.pdf"
tables = camelot.read_pdf(path, 
                          flavor='lattice', 
                          process_background=True,
                          line_scale=40,
                          pages='all')

print(f"總共偵測到 {len(tables)} 個表格區塊")

# 2. 合併所有表格 
# 注意：上銀型錄每頁的欄位結構可能略有不同（例如 FSV 與 FSI 型），
# 建議先檢查欄位數量是否一致再合併。
all_dfs = [t.df for t in tables]
full_df = pd.concat(all_dfs, ignore_index=True)

# 3. 顯示前 50 行檢查 
display(full_df.head(50))

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)


總共偵測到 4 個表格區塊


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,Form A\nTYPE 1\nTYPE 2\nL2\nL7\nL1\nM\nL11\n22...,,,,,,GG,,,,...,L8\nForm C,D6,,,,NaN,NaN,NaN,NaN,NaN
1,,,,,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN
2,,,,,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN
3,,,,,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN
4,,,,,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN
5,,D6,D4,,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN
6,,,,,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN
7,,,,D5\nL10,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN
8,,,,,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN
9,,,,,,,,,,,...,,,,,,NaN,NaN,NaN,NaN,NaN


In [ ]:
import numpy as np
def Data_cleanong(tables):
    cols_name = ["型號", "公稱 外徑", "導程", "珠徑", "PCD", "根徑", "珠卷數", "剛性 kfg/umk", "動負荷 C (kfg)", "靜負荷 Co (kfg)"]
    df = tables.df[2:]
    df = df.iloc[:, :10]
    df.columns = cols_name

        
    def split_all_merged_cells(df):
        # 建立一個副本，避免更動原始數據
        new_df = df.copy()
        
        # 遍歷每一列
        for index, row in new_df.iterrows():
            # 遍歷每一欄 (除了最後一欄，因為最後一欄沒人可以推擠)
            rows_count = len(new_df)
            cols_count = len(new_df.columns)
        for r in range(rows_count):
            for c in range(cols_count - 1): # 到倒數第二欄為止
                cell_val = new_df.iat[r, c]
                
                if '\n' in cell_val:
                    parts = cell_val.split('\n')
                    # 前半段留在原地
                    new_df.iat[r, c] = parts[0].strip()
                    # 後半段推擠到右邊那格
                    # 注意：如果右邊原本有值且非空白，這會覆蓋它。
                    # 但在上銀型錄中，被合併的右邊通常是空的。
                    new_df.iat[r, c + 1] = parts[1].strip()
                    
        return new_df

    df = split_all_merged_cells(df)
    df = df.replace(r'^\s*$', np.nan, regex=True)
    df = df.ffill()
    col = ["公稱 外徑", "導程", "動負荷 C (kfg)"]
    for col in col:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

df_1 = Data_cleanong(tables[1])
df_3 = Data_cleanong(tables[3])
df = pd.concat([df_1, df_3], axis = 0, ignore_index = True)

path = R"../data/HIWIN_Specs.xlsx"
all = pd.read_excel(path, sheet_name = "ALL")
df_all = pd.concat([all, df], axis = 0)
with pd.ExcelWriter(path, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
    df_all.to_excel(writer, sheet_name = 'ALL', index = False)
